# DeepSeek Security Vulnerability Detection Fine-tuning

This notebook fine-tunes DeepSeek model for security vulnerability detection using Lightning.ai.

## Requirements
- Lightning.ai GPU instance (A10G or better recommended)
- At least 24GB VRAM for 7B model with QLoRA

In [ ]:
# Install required packages
!pip install -q transformers>=4.40.0 datasets accelerate peft bitsandbytes trl wandb

In [ ]:
import os
import json
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer
import warnings
warnings.filterwarnings('ignore')

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Configuration

In [ ]:
# Model configuration
MODEL_NAME = 'deepseek-ai/deepseek-coder-7b-instruct-v1.5'
OUTPUT_DIR = './finesec-deepseek-vuln-detector'

# Training hyperparameters
EPOCHS = 3
BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 4
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 2048
WARMUP_RATIO = 0.03

# LoRA configuration
LORA_R = 64
LORA_ALPHA = 16
LORA_DROPOUT = 0.1

# Wandb (optional)
USE_WANDB = False
WANDB_PROJECT = 'finesec-vuln-detection'

## Load and Prepare Training Data

In [ ]:
def load_jsonl_data(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

def format_chat_template(example):
    messages = example.get('messages', [])
    formatted = ''
    for msg in messages:
        role = msg['role']
        content = msg['content']
        if role == 'system':
            formatted += f'### System:\n{content}\n\n'
        elif role == 'user':
            formatted += f'### User:\n{content}\n\n'
        elif role == 'assistant':
            formatted += f'### Assistant:\n{content}\n\n'
    return {'text': formatted.strip()}

# Load training data files
data_files = [
    'training_data_large.jsonl',
    'training_data_extended.jsonl'
]

all_data = []
for file in data_files:
    if os.path.exists(file):
        all_data.extend(load_jsonl_data(file))
        print(f'Loaded {file}')

print(f'Total training examples: {len(all_data)}')

# Create dataset
dataset = Dataset.from_list(all_data)
dataset = dataset.map(format_chat_template)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(f'Train size: {len(dataset["train"])}')
print(f'Eval size: {len(dataset["test"])}')

## Load Model with QLoRA

In [ ]:
# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16
)

model.config.use_cache = False
model.config.pretraining_tp = 1

# Prepare for k-bit training
model = prepare_model_for_kbit_training(model)

print('Model loaded successfully!')

## Configure LoRA

In [ ]:
# LoRA config
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    optim='paged_adamw_32bit',
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    group_by_length=True,
    lr_scheduler_type='cosine',
    evaluation_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=100,
    logging_steps=25,
    report_to='wandb' if USE_WANDB else 'none',
    run_name='finesec-deepseek-vuln' if USE_WANDB else None
)

## Train Model

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    peft_config=peft_config,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    tokenizer=tokenizer,
    args=training_args,
    packing=False
)

print('Starting training...')
trainer.train()
print('Training complete!')

## Save Model

In [ ]:
# Save the fine-tuned model
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Model saved to {OUTPUT_DIR}')

## Test the Model

In [ ]:
def test_vulnerability_detection(code_snippet, instruction):
    prompt = f'''### System:
You are a security expert analyzing code for vulnerabilities.

### User:
{instruction}

{code_snippet}

### Assistant:
'''
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.7,
        do_sample=True,
        top_p=0.95
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split('### Assistant:')[-1].strip()

# Test examples
test_cases = [
    {
        'code': "db.execute(f'SELECT * FROM users WHERE id = {user_id}')",
        'instruction': 'Analyze this code for SQL injection vulnerabilities'
    },
    {
        'code': 'os.system("ping " + user_input)',
        'instruction': 'Check this code for command injection'
    },
    {
        'code': 'pickle.loads(request.data)',
        'instruction': 'Identify security vulnerabilities in this code'
    }
]

for test in test_cases:
    print('='*60)
    print(f'Code: {test["code"]}')
    print(f'Instruction: {test["instruction"]}')
    print('-'*60)
    result = test_vulnerability_detection(test['code'], test['instruction'])
    print(f'Analysis:\n{result}')
    print()

## Merge and Export (Optional)

In [ ]:
# Merge LoRA weights with base model for deployment
from peft import PeftModel

MERGED_OUTPUT = './finesec-deepseek-merged'

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True
)

# Load and merge LoRA
merged_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
merged_model = merged_model.merge_and_unload()

# Save merged model
merged_model.save_pretrained(MERGED_OUTPUT)
tokenizer.save_pretrained(MERGED_OUTPUT)
print(f'Merged model saved to {MERGED_OUTPUT}')